# 🏥 Patient Readmission Prediction
## Step 1: Exploratory Data Analysis (EDA)

**Dataset**: Diabetes 130-US Hospitals (1999–2008)
**Goal**: Understand patterns that lead to 30-day readmission

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## 1.1 Load Dataset
Download from: https://www.kaggle.com/datasets/brandao/diabetic
Place `diabetic_data.csv` in the `data/` folder.

In [ ]:
df = pd.read_csv('../data/diabetic_data.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 1.2 Basic Info

In [ ]:
print('=== Dataset Info ===')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'\nData Types:\n{df.dtypes.value_counts()}')
print(f'\nMissing Values (shown as ? in this dataset):')
for col in df.columns:
    q_count = (df[col] == '?').sum()
    if q_count > 0:
        print(f'  {col}: {q_count} ({q_count/len(df)*100:.1f}%)')

## 1.3 Target Variable: Readmission

In [ ]:
# Original: '<30', '>30', 'NO' — we convert to binary
# Readmitted within 30 days = 1, else = 0
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original distribution
df['readmitted'].value_counts().plot(kind='bar', ax=axes[0], color=['#2196F3','#FF9800','#4CAF50'])
axes[0].set_title('Readmission (Original 3 Classes)', fontsize=13)
axes[0].set_xlabel('')
axes[0].tick_params(rotation=0)

# Binary distribution
counts = df['readmitted_binary'].value_counts()
axes[1].pie(counts, labels=['Not Readmitted (0)', 'Readmitted <30 days (1)'],
            autopct='%1.1f%%', colors=['#4CAF50','#F44336'], startangle=90)
axes[1].set_title('Binary Target Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('../data/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nClass Imbalance Ratio: {counts[0]/counts[1]:.1f}:1')

## 1.4 Age Distribution vs Readmission

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']

age_counts = df['age'].value_counts().reindex(age_order)
age_counts.plot(kind='bar', ax=axes[0], color='#5C6BC0')
axes[0].set_title('Patient Count by Age Group', fontsize=13)
axes[0].set_xlabel('Age Group')
axes[0].tick_params(rotation=45)

age_readmit = df.groupby('age')['readmitted_binary'].mean().reindex(age_order) * 100
age_readmit.plot(kind='bar', ax=axes[1], color='#EF5350')
axes[1].set_title('Readmission Rate (%) by Age Group', fontsize=13)
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Readmission Rate (%)')
axes[1].tick_params(rotation=45)

plt.tight_layout()
plt.savefig('../data/age_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 Hospital Stay & Medications

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

numeric_features = ['time_in_hospital', 'num_medications', 'num_lab_procedures', 'number_diagnoses']
titles = ['Time in Hospital (days)', 'Number of Medications', 'Lab Procedures', 'Number of Diagnoses']
colors = ['#42A5F5', '#66BB6A', '#FFA726', '#AB47BC']

for ax, feat, title, color in zip(axes.flat, numeric_features, titles, colors):
    for val, label in [(0, 'Not Readmitted'), (1, 'Readmitted <30d')]:
        subset = df[df['readmitted_binary'] == val][feat]
        ax.hist(subset, bins=20, alpha=0.6, label=label, density=True)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions by Readmission Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.6 Key Insights Summary

In [ ]:
print('=== KEY EDA INSIGHTS ===')
print(f'Total patients: {len(df):,}')
print(f'Readmitted <30 days: {df["readmitted_binary"].sum():,} ({df["readmitted_binary"].mean()*100:.1f}%)')
print(f'Average hospital stay: {df["time_in_hospital"].mean():.1f} days')
print(f'Average medications: {df["num_medications"].mean():.1f}')
print(f'Average lab procedures: {df["num_lab_procedures"].mean():.1f}')
print(f'Most common age group: {df["age"].mode()[0]}')
print('\n✅ EDA Complete! Proceed to 02_preprocessing.ipynb')